# Plot Spice
This is a prototyping notebook for plotting SPICE results locally using our plot framework. This is only intended for prototyping and testing purposes and should be deleted before final submission.

In [1]:
from pathlib import Path
import sys
from analysis_util import find_switching_time, get_sample, apply_time_shift

from bokeh.io import output_notebook
output_notebook()

from plot_framework import iplot, ioverlay, isweep, istack, isweep_overlay
from ngspice_loader import load_wrdata, load_sweep

PROJECT_ROOT = Path("~/CaC_Spring26")
SPICE_DIR = PROJECT_ROOT / "spice"

Loading BokehJS ...

## Experiment Block

In [ ]:
# Single-file waveform viewer
# traces = load_wrdata(
#     SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
#     ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
# )

# istack(
#     layers=[
#         {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
#         {"RST": traces["RST"]},
#         {"UP": traces["UP"], "DOWN": traces["DOWN"]},
#     ],
#     title="FF1 phase detector — clk_in leads",
#     xlabel="Time (ns)",
#     ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
# )

# print(traces)

# Plot one signal
# iplot(*traces["DOWN"], title="UP output", xlabel="Time (ns)", ylabel="V")

# Overlay all signals
# ioverlay(traces, title="FF1 — clk_in leads", xlabel="Time (ns)", ylabel="Voltage (V)")

# Overlay a subset
# ioverlay(
#     {k: traces[k] for k in ["CLK_IN", "UP", "DOWN"]},
#     title="FF1 — clock vs outputs",
#     xlabel="Time (ns)", ylabel="V",
# )

# Compare across test phases (sweep)
# sweep = load_sweep(
#     {
#         "clk_in leads":  (SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",  ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"]),
#         "clk_out leads": (SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv", ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"]),
#     },
#     signal="DOWN",
# )
# # {"clk_in leads": (t, v), "clk_out leads": (t, v), "near zero": (t, v)}
# print(sweep["clk_in leads"])
# print(sweep["clk_out leads"])

# iplot(*sweep["clk_in leads"], title="UP output", xlabel="Time (ns)", ylabel="V")

# isweep(sweep, title="DOWN across phases", xlabel="Time (ns)", ylabel="V")

# # Compare across detectors (sweep + overlay)
# groups = load_sweep_overlay(
#     sweep_axis={
#         "clk_in leads":  {
#             "FF1": ("results/ff1/clk_in_leads_results.csv",  ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#             "PFD": ("results/pfd/clk_in_leads_results.csv",  ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#         },
#         "clk_out leads": {
#             "FF1": ("results/ff1/clk_out_leads_results.csv", ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#             "PFD": ("results/pfd/clk_out_leads_results.csv", ["CLK_IN","CLK_OUT","RST","UP","DOWN"]),
#         },
#     },
#     signal="DOWN",
# )
# isweep_overlay(groups, title="DOWN: FF1 vs PFD", xlabel="Time (ns)", ylabel="V")

Loading BokehJS ...

## NAND DCDL

In [2]:
# v(A) v(Y) v(Q0_node) v(Q1_node) v(Q2_node) v(Q3_node)
traces = load_wrdata(
    SPICE_DIR / "tmp_results" / "nand_dcdl_results.csv",
    ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node"],
)

istack(
    layers=[
        {"Q0": traces["Q0_node"], "Q1": traces["Q1_node"],
         "Q2": traces["Q2_node"], "Q3": traces["Q3_node"]},
        {"A": traces["A"]},
        {"Y": traces["Y"]},
    ],
    title="NAND DCDL",
    xlabel="Time (ns)",
    ylabels=["Q[0:3] (V)", "A (V)", "Y (V)"],
)

In [10]:
# Find propagation delay
for time_interval in [(0, 25), (25, 50), (50, 75), (75, 100)]:
    t_start, t_end = time_interval
    A_switch_time = find_switching_time(traces["A"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    Y_switch_time = find_switching_time(traces["Y"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    prop_delay_LH = Y_switch_time - A_switch_time
    print(f"Low to High Propagation delay from t={t_start} to t={t_end}: {prop_delay_LH : .5f} ns")

Low to High Propagation delay from t=0 to t=25:  0.10667 ns
Low to High Propagation delay from t=25 to t=50:  0.18043 ns
Low to High Propagation delay from t=50 to t=75:  0.25253 ns
Low to High Propagation delay from t=75 to t=100:  0.32471 ns


In [11]:
delta_t = 1
sampled_traces = {}
for q_val, time_interval in ({"Q0": (0, 25), "Q1": (25, 50), "Q2": (50, 75), "Q3": (75, 100)}).items():
    t_start, t_end = time_interval
    A_switch_time = find_switching_time(traces["A"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    Y_switch_time = find_switching_time(traces["Y"], t_start=t_start, t_end=t_end, edge="rising", occurrence=1)
    sampled_trace = get_sample(traces["Y"], t_start=Y_switch_time - delta_t, t_end = Y_switch_time + delta_t)
    sampled_traces[q_val] = apply_time_shift(sampled_trace, time_shift=-A_switch_time)

ioverlay(
    {k: sampled_traces[k] for k in ["Q0", "Q1", "Q2", "Q3"]},
    title="Rising edge of ",
    xlabel="Propagation delay time relative to switch time of A (ns)", ylabel="Y (V)",
)

## Controller 2 Mode

In [9]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_2mode.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## Controller Variable Step

In [11]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_variable_step.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## Controller Saturate

In [10]:
from analysis_util import decode_ctrl

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_saturate.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller — Post-Layout SPICE",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

## FF1 Phase Detector

In [47]:
traces = load_wrdata(
    SPICE_DIR / "tmp_results" / "phase_detector_syn_edge_clkin22n_clkout20n.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [13]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

## Edge Phase Detector

In [14]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="Edge phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [16]:
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="Edge phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)